# Chapitre 8 — Mesurer la ressemblance · Notebook de travail

*Maths pour l'IA, pas à pas*, Belk Siri — notebook compagnon.

[Ouvrir dans Google Colab](https://colab.research.google.com/github/belksiri/books/blob/main/maths-pour-l-ia-pas-a-pas/notebooks/fr/ch08_a_completer.ipynb) · Page compagnon : https://belksiri.github.io/books/maths-pour-l-ia-pas-a-pas/

Les cellules à modifier portent la mention **À COMPLÉTER**. Exécutez les cellules dans l'ordre : tant qu'une réponse manque, les vérifications le signalent sans erreur. Le corrigé se trouve dans `ch08_corrige.ipynb`, sur la même page compagnon.

**Sommaire.** Données · Les calculs des leçons 8.1 à 8.3 · Exercices 8.1 à 8.6

**Avant d'ouvrir Jupyter :** les exercices de ce chapitre se font d'abord sur papier, avec une calculatrice. Les cellules ci-dessous servent à **vérifier** vos calculs. Comme dans le livre, moyennes et écarts-types divisent par $n$ et se calculent sur les six ventes de référence, A à F.

**Noms des variables.** Le code emploie les noms anglais du livre : `floor_area` (surface, en m²), `distance` (distance au centre, en km), `price` (prix, en k€), `home` (nom du logement).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Couleurs du livre : logements de référence en bleu, voisins retenus cerclés d'orange, distances en magenta
BLUE, ORANGE, MAGENTA = '#2a78d6', '#eb6834', '#c2185b'

## Données

Les identifiants D2.1 et D2.E2 sont ceux du livre et des fichiers CSV. Les colonnes sont renommées en anglais à la lecture ; la correspondance figure en commentaire.

In [ ]:
# Données du chapitre 8, recopiées des fichiers CSV du livre (ne pas modifier)

# D2.1 — les sept logements ; colonnes du CSV renommées en anglais :
#   home = logement, floor_area = surface_m2 (m²), distance = distance_km (km), price = prix_kEUR (k€)
homes = pd.DataFrame({
    'home':       ['A', 'B', 'C', 'D', 'E', 'F', 'G'],
    'floor_area': [32, 45, 54, 70, 85, 98, 62],
    'distance':   [3.0, 1.5, 2.5, 1.0, 2.5, 0.5, 2.5],
    'price':      [118, 172, 185, 245, 279, 335, 209],
})
reference = homes[homes['home'] != 'G'].reset_index(drop=True)   # les six ventes de référence, A à F
X = reference[['floor_area', 'distance']].to_numpy(dtype=float)  # une ligne par vente : (m² ; km)
price = reference['price'].to_numpy(dtype=float)                 # prix des ventes de référence, en k€
G = np.array([62, 2.5])                                          # le logement G : 62 m², 2,5 km (vendu 209 k€)

# D2.E2 — le logement I (exercice 8.2)
I = np.array([75, 3.5])   # 75 m², 3,5 km

# Exemples du livre (leçons 8.1 et 8.3) : plan de la ville en km, messages, plongements inventés
school, G_map = np.array([4.5, 6]), np.array([1.5, 2])            # l'école et G sur le plan, en km
R1, R2, P = np.array([3, 1]), np.array([6, 2]), np.array([1, 3])  # (« réunion » ; « offre »)
embeddings = {'réunion': np.array([4, 3]), 'assemblée': np.array([3, 4]),
              'soldes': np.array([-3, 4]), 'promotion': np.array([-4, 3])}

### Les variables des données

Les noms de code sont en anglais. Pour les colonnes, la dernière colonne donne le nom utilisé dans les fichiers CSV du livre.

| Nom | Sens | Unité | Colonne du CSV |
|:--|:--|:--|:--|
| `home` | identifiant du logement (A, B…) | — | `logement` |
| `floor_area` | surface habitable du logement | m² | `surface_m2` |
| `price` | prix de vente | k€ | `prix_kEUR` |
| `distance` | distance au centre-ville | km | `distance_km` |
| `distance_m` | distance au centre-ville | m | — |
| `message` | identifiant du texte (T1 à T4) | — | — |
| `meeting_count` | nombre d'occurrences du mot « réunion » | occurrences | — |
| `offer_count` | nombre d'occurrences du mot « offre » | occurrences | — |
| `volunteer_count` | nombre d'occurrences du mot « bénévole » | occurrences | — |

In [ ]:
# Outils fournis : vous n'avez pas à les modifier

def nearest(dists, k=3):
    """Les k ventes de référence les plus proches (distances données dans l'ordre A à F) et leur prix moyen."""
    order = np.argsort(dists, kind='stable')[:k]
    return list(reference['home'].iloc[order]), float(price[order].mean())


def show_table(dists, label):
    """Tableau des distances à un logement, de la plus courte à la plus longue."""
    t = pd.DataFrame({'home': reference['home'], label: np.round(dists, 3), 'price': price})
    print(t.sort_values(label, kind='stable').to_string(index=False))

## Les calculs des leçons, en Python

Ces cellules refont les calculs des leçons 8.1 à 8.3. Exécutez-les et comparez avec le livre.

In [ ]:
# Leçon 8.1 — norme et distance (équation 8.1)
trip = school - G_map                       # déplacement de G à l'école, en km
print(f"déplacement {trip}, distance {np.sqrt((trip ** 2).sum()):g} km")
print(f"du centre à G : {np.linalg.norm(G_map):g} km")             # la distance au centre du tableau 2.1
print(f"norme de (2 ; 3 ; 6) : {np.linalg.norm([2, 3, 6]):g}")      # Pythagore deux fois, ou un seul calcul
C_home = X[2]                                                       # C : 54 m², 2,5 km
print(f"distance de C à G : {np.linalg.norm(C_home - G):g} (unités mêlées : m² et km)")

In [ ]:
# Leçon 8.2 — premier essai : distances brutes à G, puis distance au centre en mètres
raw_G = np.sqrt(((X - G) ** 2).sum(axis=1))
show_table(raw_G, 'raw_distance')
n_km, e_km = nearest(raw_G)
print(f"3 plus proches voisins (brut, km) : {n_km}, estimation {e_km:.1f} k€")
X_m, G_m = X * [1, 1000], G * [1, 1000]                             # distance au centre en mètres
n_m, e_m = nearest(np.linalg.norm(X_m - G_m, axis=1))
print(f"3 plus proches voisins (brut, m)  : {n_m}, estimation {e_m:.0f} k€")

In [ ]:
# Leçon 8.2 — deuxième essai : standardiser avec les ventes de référence (équation 8.2)
mu, sigma = X.mean(axis=0), X.std(axis=0)                           # division par n (ddof=0)
print(f"surface : moyenne {mu[0]:.0f} m², écart-type {sigma[0]:.2f} m²")
print(f"distance : moyenne {mu[1]:.3f} km, écart-type {sigma[1]:.3f} km (√29/6 = {np.sqrt(29) / 6:.3f})")
Z, z_G = (X - mu) / sigma, (G - mu) / sigma
print('z de G :', np.round(z_G, 2))
std_G = np.linalg.norm(Z - z_G, axis=1)
show_table(std_G, 'std_distance')
neighbours, estimate = nearest(std_G)
print(f"3 plus proches voisins (standardisé) : {neighbours}, estimation {estimate:.0f} k€ (prix réel : 209 k€)")

fig, ax = plt.subplots(figsize=(5, 4))
ax.scatter(Z[:, 0], Z[:, 1], color=BLUE, s=40)
ax.scatter(*z_G, marker='D', color='black', s=40)
for h, (a, b) in zip(reference['home'], Z):
    ax.annotate(h, (a, b), xytext=(5, 4), textcoords='offset points')
ax.annotate('G', z_G, xytext=(5, 4), textcoords='offset points')
idx = np.argsort(std_G, kind='stable')[:3]
ax.scatter(Z[idx, 0], Z[idx, 1], s=220, facecolors='none', edgecolors=ORANGE, linewidths=1.8)
ax.add_patch(plt.Circle(z_G, std_G[idx[-1]], fill=False, linestyle='--', color='grey'))
ax.set_aspect('equal'); ax.set_xlabel('z surface'); ax.set_ylabel('z distance'); ax.grid(alpha=0.4)
plt.show()

In [ ]:
# Leçon 8.3 — distance ou cosinus ? (équation 8.3)
def cos_lesson(u, v):
    return float(np.dot(u, v) / (np.linalg.norm(u) * np.linalg.norm(v)))

print(f"distances : R1–R2 {np.linalg.norm(R1 - R2):.2f}, R1–P {np.linalg.norm(R1 - P):.2f}")
print(f"cosinus   : R1–R2 {cos_lesson(R1, R2):.2f}, R1–P {cos_lesson(R1, P):.2f}, "
      f"angle R1–P {np.degrees(np.arccos(cos_lesson(R1, P))):.0f}°")

# Aperçu : les plongements du tableau 8.3 (tous de norme 5)
for a, b in [('réunion', 'assemblée'), ('soldes', 'promotion'), ('réunion', 'soldes'), ('réunion', 'promotion')]:
    print(f"cos({a}, {b}) = {cos_lesson(embeddings[a], embeddings[b]):.2f}")
message = (embeddings['réunion'] + embeddings['assemblée']) / 2       # le message « réunion, assemblée »
print(f"message {message} : cos avec réunion {cos_lesson(message, embeddings['réunion']):.2f}, "
      f"avec promotion {cos_lesson(message, embeddings['promotion']):.2f}")

## Exercice 8.1 — Distances sur le plan de la ville

*Notions : norme et distance (leçon 8.1, équation 8.1).* Sur le plan de la figure 8.1, en kilomètres, la piscine se trouve au point $(7{,}5 ; 10)$ et la gare au point $(-3{,}5 ; 5)$ ; G est au point $(1{,}5 ; 2)$.

### a) à e) Sur papier

*Vos calculs :* …

In [ ]:
# À COMPLÉTER : la norme d'un vecteur, sans np.linalg.norm (équation 8.1)
def norm(u):
    u = np.asarray(u, dtype=float)
    return None   # À COMPLÉTER : racine carrée de la somme des carrés des coordonnées

pool, station = np.array([7.5, 10]), np.array([-3.5, 5])   # piscine et gare, en km

In [ ]:
# Vérification de a) à e)
if norm([6, 8]) is None:
    print('pas encore complété. Complétez la fonction norm.')
else:
    d_pool, d_station, d_station_pool = norm(pool - G_map), norm(station - G_map), norm(pool - station)
    print(f"a) {norm([6, 8]):g}   b) {d_pool:g} km   c) {d_station:.2f} km et {d_station_pool:.2f} km")
    print(f"d) détour : {d_station + d_station_pool - d_pool:.2f} km   e) {norm([2, 3, 6]):g}")
    print('→', 'correct' if abs(d_pool - 10) < 1e-9 and abs(d_station - 5.831) < 5e-4
          and abs(d_station_pool - 12.083) < 5e-4 and abs(norm([2, 3, 6]) - 7) < 1e-9 else 'à revoir')

## Exercice 8.2 — Les voisins du logement I

*Notions : distance brute et distance standardisée (leçon 8.2, équation 8.2).* Le logement I (jeu D2.E2) a pour `floor_area` 75 m² et pour `distance` 3,5 km : c'est le vecteur `I`. On l'estime avec les six ventes de référence et $k = 3$.

- `floor_area` — surface du logement, en m² (colonne `surface_m2` du CSV) ;
- `distance` — distance au centre, en km (colonne `distance_km` du CSV) ;
- `price` — prix des ventes de référence, en k€ ; scores $z$ sans unité.

### a) à e) Sur papier

*Vos calculs et vos interprétations :* …

In [ ]:
# À COMPLÉTER : distances de I aux six ventes A à F, brutes puis standardisées
raw_I = None   # À COMPLÉTER : np.sqrt(((X - I) ** 2).sum(axis=1))
z_I = None     # À COMPLÉTER : scores de I avec mu et sigma (leçon 8.2)
std_I = None   # À COMPLÉTER : distances entre les lignes de Z et z_I

In [ ]:
# Vérification de a) à c)
if raw_I is None or z_I is None or std_I is None:
    print('pas encore complété. Complétez les trois lignes.')
else:
    show_table(raw_I, 'raw_distance')
    (n_raw, e_raw), (n_std, e_std) = nearest(raw_I), nearest(std_I)
    print(f"a) brut : {n_raw}, estimation {e_raw:.1f} k€")
    print(f"b) z de I : {np.round(z_I, 2)}")
    show_table(std_I, 'std_distance')
    print(f"c) standardisé : {n_std}, estimation {e_std:.0f} k€")
    print('→', 'correct' if n_raw == ['D', 'E', 'C'] and abs(e_raw - 236.33) < 0.01
          and n_std == ['E', 'C', 'A'] and abs(e_std - 194) < 1e-9 else 'à revoir')

## Exercice 8.3 — Mètres ou kilomètres ?

*Notions : distance brute et unités (leçon 8.2).* On reprend G et les six ventes, mais la distance au centre est comptée en mètres : `distance_m` $= 1\,000 \times$ `distance`.

- `distance_m` — distance au centre, en m ;
- `distance` — distance au centre, en km ; `floor_area` (m²) et `price` (k€) des ventes de référence.

### a) à d) Sur papier

*Vos calculs et vos réponses :* …

In [ ]:
# À COMPLÉTER : la distance au centre en mètres, puis les distances brutes à G
distance_m = None     # À COMPLÉTER : 1000 × la colonne distance des ventes A à F
G_distance_m = None   # À COMPLÉTER : distance de G au centre, en m
raw_G_m = None        # À COMPLÉTER : distances brutes à G avec (floor_area ; distance_m)

In [ ]:
# Vérification de a), c) et d)
if distance_m is None or G_distance_m is None or raw_G_m is None:
    print('pas encore complété. Complétez les trois lignes.')
else:
    show_table(raw_G_m, 'raw_distance_m')
    n_m, e_m = nearest(raw_G_m)
    print(f"a) en mètres : {n_m}, estimation {e_m:.0f} k€")
    mu_m, sigma_m = distance_m.mean(), distance_m.std()
    z_G_m = (G_distance_m - mu_m) / sigma_m
    print(f"c) moyenne {mu_m:.0f} m, écart-type {sigma_m:.0f} m ; z de G : {z_G_m:.2f} en m, {z_G[1]:.2f} en km")
    print('→', 'correct' if n_m == ['C', 'E', 'A'] and abs(e_m - 194) < 1e-9 and abs(z_G_m - z_G[1]) < 1e-12
          else 'à revoir')

## Exercice 8.4 — Combien de voisins ?

*Notions : plus proches voisins, choix de $k$ (leçon 8.2), validation (leçon 6.4).* Dans l'ordre des distances standardisées à G, les six ventes et leur `price` sont C (185 k€), E (279), B (172), A (118), D (245) et F (335).

- `price` — prix de vente, en k€ ; $k$ — nombre de voisins retenus, sans unité.

### a) à c) Sur papier

*Vos estimations et votre réponse :* …

In [ ]:
# À COMPLÉTER : l'estimation de G pour chaque valeur de k
estimates = None   # À COMPLÉTER : {k: nearest(std_G, k)[1] for k in (1, 2, 3, 4, 6)}

In [ ]:
# Vérification de a) et c)
if estimates is None:
    print('pas encore complété. Complétez la ligne.')
else:
    for k, e in estimates.items():
        print(f"k = {k} : {e:.1f} k€, écart avec le prix réel (209 k€) : {e - 209:+.1f} k€")
    expected = {1: 185, 2: 232, 3: 212, 4: 188.5, 6: 1334 / 6}
    print('→', 'correct' if all(abs(estimates[k] - v) < 1e-9 for k, v in expected.items()) else 'à revoir')

## Exercice 8.5 — Distance ou cosinus ?

*Notions : distance (équation 8.1), cosinus (équation 8.3).* On compte trois mots dans quatre messages de l'association : « réunion », « offre » et « bénévole » (tableau 8.4 du livre).

- `message` — identifiant du message, T1 à T4 ;
- `meeting_count`, `offer_count`, `volunteer_count` — nombres d'emplois des mots « réunion », « offre » et « bénévole » dans le message : les trois coordonnées de son vecteur.

In [ ]:
# Données de l'exercice 8.5 (tableau du livre)
words = pd.DataFrame({
    'message':         ['T1', 'T2', 'T3', 'T4'],
    'meeting_count':   [2, 6, 1, 0],
    'offer_count':     [0, 0, 2, 1],
    'volunteer_count': [1, 3, 2, 0],
}).set_index('message')
T = {m: words.loc[m].to_numpy(dtype=float) for m in words.index}
print(words)

### a) à d) Sur papier

*Vos calculs et votre interprétation :* …

In [ ]:
# À COMPLÉTER : le cosinus de deux vecteurs (équation 8.3)
def cosine(u, v):
    return None   # À COMPLÉTER : produit scalaire divisé par le produit des normes

In [ ]:
# Vérification de a) à c)
if cosine(T['T1'], T['T2']) is None:
    print('pas encore complété. Complétez la fonction cosine.')
else:
    results = pd.DataFrame({
        'norm': [np.linalg.norm(T[m]) for m in T],
        'distance_to_T1': [np.linalg.norm(T['T1'] - T[m]) for m in T],
        'cosine_with_T1': [cosine(T['T1'], T[m]) for m in T],
    }, index=list(T)).round(2)
    print(results)
    print('→', 'correct' if abs(cosine(T['T1'], T['T2']) - 1) < 1e-12 and abs(cosine(T['T1'], T['T3']) - 0.596) < 5e-4
          and abs(cosine(T['T1'], T['T4'])) < 1e-12 else 'à revoir')

## Exercice 8.6 — Défi : normaliser, puis comparer

*Notions : norme et produit scalaire (équation 8.1), cosinus (équation 8.3).* Les questions a) et b) sont des démonstrations sur papier ; le notebook vérifie c) et d).

### a) à d) Sur papier

*Vos démonstrations et vos calculs :* …

In [ ]:
# À COMPLÉTER : chaque message divisé par sa norme, puis les distances à T1 normalisé
normalized = None     # À COMPLÉTER : {m: T[m] / np.linalg.norm(T[m]) for m in T}
unit_dist = None      # À COMPLÉTER : {m: distance entre normalized['T1'] et normalized[m] for m in T}

In [ ]:
# Vérification de c) et d)
if normalized is None or unit_dist is None:
    print('pas encore complété. Complétez les deux lignes.')
else:
    for m in ('T2', 'T3', 'T4'):
        c = float(np.dot(T['T1'], T[m]) / (np.linalg.norm(T['T1']) * np.linalg.norm(T[m])))
        print(f"{m} : distance normalisée {unit_dist[m]:.2f}, √(2 − 2 cos) = {np.sqrt(max(2 - 2 * c, 0)):.2f}")
    u, v = embeddings['réunion'] / 5, embeddings['assemblée'] / 5
    print(f"d) réunion et assemblée : directement {np.linalg.norm(u - v):.2f}, avec b) {np.sqrt(2 - 2 * np.dot(u, v)):.2f}")
    print('→', 'correct' if abs(unit_dist['T2']) < 1e-9 and abs(unit_dist['T3'] - 0.899) < 5e-4
          and abs(unit_dist['T4'] - np.sqrt(2)) < 1e-9 else 'à revoir')